## Workspace setup

In [ ]:
from datetime import datetime 
import uproot
import awkward as ak
import tensorflow as tf
import numpy as np
import importlib
from functools import partial

from tensorflow.data import Dataset, TFRecordDataset
from tensorflow.data.experimental import TFRecordWriter
from tensorflow.train import BytesList, FloatList, Int64List
from tensorflow.train import Example, Features, Feature
import tensorflow_datasets as tfds

#Notebooks run from VSCode use home directory as a base path
#while notebooks run from JupyterLab use the current directory as a base path
import sys
sys.path.append("/home/akalinow/scratch/ELITPC/TPCReco/PythonAnalysis/python/")

import io_functions as io

### Load and convert SimEvent data

In [ ]:
%%time
importlib.reload(io)

dataPath = '/home/akalinow/scratch/ELITPC/TPCReco/build/resources/'
rootfiles = [dataPath+'SimEvent_Track3D_TwoProng_gun_MC.root:TPCData']
outputDir = 'SimEvent_Track3D_TwoProng_gun_MC'

# Convert ROOT files to TF format and save to output directory
io.convertROOT(rootfiles, outputDir, fields= io.simEventFields)

# Load the dataset and benchmark it
batchSize = 32
simDataset = tf.data.Dataset.load(outputDir, compression="GZIP")
simDataset = dataset.batch(batchSize).cache()

tfds.benchmark(simDataset, batch_size=batchSize)

# Second pass profiting from the cache
tfds.benchmark(simDataset, batch_size=batchSize)

### Load and convert RecoEvent data

In [ ]:
%%time
importlib.reload(io)

dataPath = '/home/akalinow/scratch/ELITPC/TPCReco/build/resources/'
# uproot always takes the first branch with given name, unless 
# explicit branch is givent as input path. Filtering by branches in
# iterate does not work. 
rootfiles = [dataPath+'SimEvent_Track3D_TwoProng_gun_MC.root:TPCData/RecoEvent']
outputDir = 'RecoEvent_Track3D_TwoProng_gun_MC'

# Convert ROOT files to TF format and save to output directory
io.convertROOT(rootfiles, outputDir, fields=io.recoEventFields)

# Load the dataset and benchmark it
batchSize = 32
recoDataset = tf.data.Dataset.load(outputDir, compression="GZIP")
recoDataset = dataset.batch(batchSize).cache()

tfds.benchmark(recoDataset, batch_size=batchSize)

# Second pass profiting from the cache
tfds.benchmark(recoDataset, batch_size=batchSize)

### Merge SimEvent and RecoEvent data


In [ ]:
# merge SimEvent and RecoEvent data
simDataset = tf.data.Dataset.load('SimEvent_Track3D_TwoProng_gun_MC', compression="GZIP")
recoDataset = tf.data.Dataset.load('RecoEvent_Track3D_TwoProng_gun_MC', compression="GZIP")

mergedDataset = tf.data.Dataset.zip((simDataset, recoDataset))
mergedDataset = mergedDataset.map(
    lambda sim, reco: {
        'sim': sim,
        'reco': reco
    }
)
# save merged dataset
outputDir = 'MergedEvent_Track3D_TwoProng_gun_MC'
mergedDataset.save(outputDir, compression="GZIP")

### Plot a test event

In [ ]:
import plotting_functions as plf
importlib.reload(plf)

x = iter(simDataset.batch(1))
plf.plotEvent(item, model=None)